# Emoji-Based Mood Analyzer
**Ekman 6-Emotion Classification using Emoji + Text Features**

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter

from model.train import train, build_feature_matrix, EMOJI_SENTIMENT
from model.predict import load_model, predict, check_wellbeing, MOOD_META

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'
print('Setup complete')

## 1. Load & Explore Dataset

In [ ]:
df = pd.read_csv('../data/sample_data.csv')
print(f'Shape: {df.shape}')
print('\nClass distribution:')
print(df['mood'].value_counts())
df.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
mood_counts = df['mood'].value_counts()
colors = [MOOD_META.get(m, {}).get('color', '#aaa') for m in mood_counts.index]
ax.bar(mood_counts.index, mood_counts.values, color=colors, width=0.6)
ax.set_title('Mood class distribution', fontsize=13)
ax.set_ylabel('Count')
ax.spines[['top', 'right']].set_visible(False)
for i, (m, v) in enumerate(zip(mood_counts.index, mood_counts.values)):
    ax.text(i, v + 0.3, str(v), ha='center', fontsize=10)
plt.tight_layout()
plt.show()

## 2. Feature Engineering — Emoji Analysis

In [ ]:
features_df = build_feature_matrix(df)
print('Feature matrix shape:', features_df.shape)
print('\nFeature columns:', list(features_df.columns))
features_df[['original_text', 'mood', 'emoji_count', 'sentiment_score', 'dominant_emoji_mood']].head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Avg emoji count per mood
avg_emoji = features_df.groupby('mood')['emoji_count'].mean().sort_values(ascending=False)
clrs = [MOOD_META.get(m, {}).get('color', '#aaa') for m in avg_emoji.index]
axes[0].bar(avg_emoji.index, avg_emoji.values, color=clrs, width=0.6)
axes[0].set_title('Avg emoji count per mood')
axes[0].set_ylabel('Avg count')
axes[0].spines[['top','right']].set_visible(False)

# Sentiment score distribution
for mood in features_df['mood'].unique():
    subset = features_df[features_df['mood'] == mood]['sentiment_score']
    color = MOOD_META.get(mood, {}).get('color', '#aaa')
    axes[1].hist(subset, bins=8, alpha=0.5, label=mood, color=color)
axes[1].set_title('Sentiment score distribution by mood')
axes[1].set_xlabel('Sentiment score')
axes[1].legend(fontsize=8)
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.show()

## 3. Emoji Lexicon — Most Informative Signals

In [ ]:
lex_df = pd.DataFrame([
    {'emoji': e, 'mood': m, 'weight': w}
    for e, (m, w) in EMOJI_SENTIMENT.items()
]).sort_values(['mood', 'weight'], ascending=[True, False])

fig, ax = plt.subplots(figsize=(10, 4))
for mood, grp in lex_df.groupby('mood'):
    top = grp.nlargest(4, 'weight')
    color = MOOD_META.get(mood, {}).get('color', '#888')
    labels = [row['emoji'] for _, row in top.iterrows()]
    x_pos = [list(lex_df['mood'].unique()).index(mood) * 6 + i for i in range(len(labels))]
    ax.bar(x_pos, top['weight'].values, color=color, width=0.7, alpha=0.85)
    for xp, lbl in zip(x_pos, labels):
        ax.text(xp, -0.08, lbl, ha='center', fontsize=14)

ax.set_xticks([list(lex_df['mood'].unique()).index(m)*6+1.5 for m in lex_df['mood'].unique()])
ax.set_xticklabels(lex_df['mood'].unique(), fontsize=10)
ax.set_ylabel('Lexicon weight')
ax.set_title('Top emoji signals per mood category')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

## 4. Train & Evaluate Models

In [ ]:
artifacts = train('../data/sample_data.csv', save_dir='../model/artifacts')

In [ ]:
# Model comparison bar chart
summary = artifacts['results_summary']
names = list(summary.keys())
cv_scores = [summary[k]['cv_f1'] for k in names]
test_scores = [summary[k]['test_f1'] for k in names]

x = np.arange(len(names))
fig, ax = plt.subplots(figsize=(9, 4))
b1 = ax.bar(x - 0.2, cv_scores,  0.38, label='CV F1',   color='#534AB7', alpha=0.85)
b2 = ax.bar(x + 0.2, test_scores, 0.38, label='Test F1', color='#1D9E75', alpha=0.85)
ax.bar_label(b1, fmt='%.3f', padding=3, fontsize=8)
ax.bar_label(b2, fmt='%.3f', padding=3, fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=9)
ax.set_ylim(0, 1.12)
ax.set_ylabel('F1 Score (macro)')
ax.set_title('Model comparison — CV vs test F1')
ax.legend()
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix
cm = artifacts['confusion_matrix']
classes = artifacts['classes']
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Confusion matrix — {artifacts["model_name"]}')
plt.tight_layout()
plt.show()

## 5. Live Inference Examples

In [ ]:
test_cases = [
    'Got promoted today!! 😊🎉✨',
    'Missing my family so much 😢💔',
    'They lied to my face again 😡🤬',
    'Waiting for biopsy results 😰😨',
    'Wait WHAT double salary offer 😲🤯',
    'That behaviour is nauseating 🤮😒',
]

for text in test_cases:
    r = predict(text, artifacts)
    meta = MOOD_META.get(r['predicted_mood'], {})
    print(f"{meta.get('icon','?')}  [{r['predicted_mood'].upper():<9} {r['confidence']:5.1f}%]  {text}")

## 6. Well-being Alert System Demo

In [ ]:
# Simulate a user going through a rough patch
session = [
    'Had a great morning 😊',
    'Feeling sad about the news 😢',
    'So angry about this situation 😡',
    'Scared about what comes next 😨',
    'Still feeling really low 😔💔',
]

mood_seq = [predict(t, artifacts)['predicted_mood'] for t in session]
print('Predicted moods:', mood_seq)

wb = check_wellbeing(mood_seq)
print('\nWell-being check:')
for k, v in wb.items():
    print(f'  {k}: {v}')